# Phase 4: Linear Least-Squares Population Decoder

OLS decoder: X -> (cos(2*theta), sin(2*theta)) -> theta_pred


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC = os.path.join(BASE, 'data', 'processed')
FINAL = os.path.join(BASE, 'data', 'final')
TABLES = os.path.join(BASE, 'reports', 'tables')


## Load Data

In [ ]:
d = np.load(os.path.join(PROC, 'orientation_decoder_ready_top1000.npz'))
X = d['X']; y_cos2 = d['y_cos2']; y_sin2 = d['y_sin2']
theta_deg = d['theta_deg']; theta_rad = d['theta_rad']
print(f'X: {X.shape}, trials: {len(theta_deg)}')


## Simple Decoder Demo

In [ ]:
rng = np.random.default_rng(42)
n = X.shape[0]
idx = rng.permutation(n)
split = int(0.8 * n)
train, test = idx[:split], idx[split:]

Y_train = np.column_stack([y_cos2[train], y_sin2[train]])
W, _, _, _ = np.linalg.lstsq(X[train], Y_train, rcond=None)
Y_pred = X[test] @ W
pred_rad = np.arctan2(Y_pred[:, 1], Y_pred[:, 0]) / 2 % np.pi
true_rad = theta_rad[test]
err = (pred_rad - true_rad + np.pi/2) % np.pi - np.pi/2
mae = float(np.degrees(np.abs(err).mean()))
print(f'MAE = {mae:.1f} deg (chance ~ 45 deg)')


## Neuron Count Scaling

In [ ]:
cv_summary = pd.read_csv(os.path.join(TABLES, 'decoder_cv_summary.csv'))
cv_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(cv_summary['neuron_count'], cv_summary['mae_mean'],
            yerr=cv_summary['mae_std'].fillna(0), fmt='o-', capsize=4, lw=2)
ax.axhline(45, color='gray', ls=':', label='Chance')
ax.set_xlabel('Neurons'); ax.set_ylabel('MAE (deg)')
ax.set_title('Decoder scaling'); ax.set_xscale('log')
ax.legend(); plt.tight_layout(); plt.show()


## Predicted vs True

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(np.degrees(true_rad), np.degrees(pred_rad), s=1, alpha=0.3)
ax.plot([0,180],[0,180],'r--',lw=1)
ax.set_xlabel('True (deg)'); ax.set_ylabel('Predicted (deg)')
ax.set_title(f'MAE = {mae:.1f} deg'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()
